In [1]:
import pandas as pd

train = pd.read_csv(r"C:\Projects\Demand-Forecasting-Analysis\Data\train.csv")

print(train.shape)
print(train.dtypes)
train.head()

(3000888, 6)
id               int64
date            object
store_nbr        int64
family          object
sales          float64
onpromotion      int64
dtype: object


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [2]:
train['date'] = pd.to_datetime(train['date'])

print("Date range:", train['date'].min(), "to", train['date'].max())
print("Number of stores:", train['store_nbr'].nunique())
print("Number of product families:", train['family'].nunique())
print("\nNulls:\n", train.isnull().sum())
print("\nDuplicate rows:", train.duplicated().sum())
print("\nSales stats:\n", train['sales'].describe())
print("\nRows with zero sales:", (train['sales'] == 0).sum())

Date range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00
Number of stores: 54
Number of product families: 33

Nulls:
 id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

Duplicate rows: 0

Sales stats:
 count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

Rows with zero sales: 939130


In [3]:
train = train.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

# Naive forecast: predict each day's sales using the same store+family's sales from exactly 7 days earlier
train['naive_forecast'] = train.groupby(['store_nbr', 'family'])['sales'].shift(7)

# Drop the first 7 days of each store+family group — no prior week to forecast from yet
train_forecast = train.dropna(subset=['naive_forecast']).copy()

train_forecast['forecast_error'] = train_forecast['sales'] - train_forecast['naive_forecast']
train_forecast['abs_error'] = train_forecast['forecast_error'].abs()

print(train_forecast.shape)
train_forecast.head(10)

(2988414, 9)


,id,date,store_nbr,family,sales,onpromotion,naive_forecast,forecast_error,abs_error
7,12474,2013-01-08,1,AUTOMOTIVE,2.0,0,0.0,2.0,2.0
8,14256,2013-01-09,1,AUTOMOTIVE,2.0,0,2.0,0.0,0.0
9,16038,2013-01-10,1,AUTOMOTIVE,2.0,0,3.0,-1.0,1.0
10,17820,2013-01-11,1,AUTOMOTIVE,3.0,0,3.0,0.0,0.0
11,19602,2013-01-12,1,AUTOMOTIVE,2.0,0,5.0,-3.0,3.0
12,21384,2013-01-13,1,AUTOMOTIVE,2.0,0,2.0,0.0,0.0
13,23166,2013-01-14,1,AUTOMOTIVE,2.0,0,0.0,2.0,2.0
14,24948,2013-01-15,1,AUTOMOTIVE,1.0,0,2.0,-1.0,1.0
15,26730,2013-01-16,1,AUTOMOTIVE,1.0,0,2.0,-1.0,1.0
16,28512,2013-01-17,1,AUTOMOTIVE,1.0,0,2.0,-1.0,1.0


In [4]:
stores = pd.read_csv(r"C:\Projects\Demand-Forecasting-Analysis\Data\stores.csv")

train_final = train_forecast.merge(stores, on='store_nbr', how='left')

print(train_final.shape)
train_final.head()

(2988414, 13)


,id,date,store_nbr,family,sales,onpromotion,naive_forecast,forecast_error,abs_error,city,state,type,cluster
0,12474,2013-01-08,1,AUTOMOTIVE,2.0,0,0.0,2.0,2.0,Quito,Pichincha,D,13
1,14256,2013-01-09,1,AUTOMOTIVE,2.0,0,2.0,0.0,0.0,Quito,Pichincha,D,13
2,16038,2013-01-10,1,AUTOMOTIVE,2.0,0,3.0,-1.0,1.0,Quito,Pichincha,D,13
3,17820,2013-01-11,1,AUTOMOTIVE,3.0,0,3.0,0.0,0.0,Quito,Pichincha,D,13
4,19602,2013-01-12,1,AUTOMOTIVE,2.0,0,5.0,-3.0,3.0,Quito,Pichincha,D,13


In [5]:
train_final.to_csv(r"C:\Projects\Demand-Forecasting-Analysis\Exports\demand_forecast_clean.csv", index=False)
print("Saved:", train_final.shape)

Saved: (2988414, 13)
